# House Prices: Advanced Regression Techniques

**Kaggle Competition**: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques

**Goal**: Predict the final sale price of each house based on 79 explanatory variables.

**Evaluation metric**: Root Mean Squared Error (RMSE) on the log of the predicted and actual sale prices.

---

## Pipeline Overview
1. Data Loading & Overview
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing & Feature Engineering
4. Model Training & Cross-Validation
5. Model Evaluation & Ensemble
6. Generate Submission

## 1. Data Loading & Overview

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import norm, skew

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

import xgboost as xgb
import lightgbm as lgb

pd.set_option('display.max_columns', 100)
sns.set_style('whitegrid')
%matplotlib inline

print('Libraries loaded successfully.')

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')

In [ ]:
train.head()

In [ ]:
train.describe()

## 2. Exploratory Data Analysis (EDA)

### 2.1 Target Variable – SalePrice

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original distribution
sns.histplot(train['SalePrice'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('SalePrice Distribution')

# Log-transformed distribution
sns.histplot(np.log1p(train['SalePrice']), kde=True, ax=axes[1], color='coral')
axes[1].set_title('Log(SalePrice + 1) Distribution')

plt.tight_layout()
plt.show()

print(f"Skewness (original)     : {train['SalePrice'].skew():.4f}")
print(f"Skewness (log-transform): {np.log1p(train['SalePrice']).skew():.4f}")

In [ ]:
# Apply log transformation to the target
train['SalePrice'] = np.log1p(train['SalePrice'])

### 2.2 Missing Values

In [ ]:
def missing_summary(df, label=''):
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    missing_pct = (missing / len(df) * 100).round(2)
    result = pd.DataFrame({'Missing': missing, 'Percent (%)': missing_pct})
    print(f'\n{label} – {len(missing)} columns with missing values')
    return result

train_missing = missing_summary(train, 'Train')
test_missing  = missing_summary(test,  'Test')

train_missing.head(20)

In [ ]:
# Visualise top missing columns
top_missing = train_missing.head(20)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_missing['Percent (%)'], y=top_missing.index, palette='Reds_r')
plt.title('Top 20 Columns with Missing Values (Train)')
plt.xlabel('Missing (%)')
plt.tight_layout()
plt.show()

### 2.3 Correlation Heatmap

In [ ]:
# Top 15 numerical features most correlated with SalePrice
corr = train.select_dtypes(include=[np.number]).corr()
top_corr = corr['SalePrice'].abs().sort_values(ascending=False).head(16).index

plt.figure(figsize=(12, 10))
sns.heatmap(
    train[top_corr].corr(),
    annot=True, fmt='.2f', cmap='coolwarm',
    square=True, linewidths=0.5
)
plt.title('Correlation Heatmap – Top 15 Features vs SalePrice')
plt.tight_layout()
plt.show()

### 2.4 Key Feature Visualisations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

pairs = [
    ('GrLivArea',    'Living Area (sq ft)'),
    ('TotalBsmtSF',  'Total Basement (sq ft)'),
    ('GarageArea',   'Garage Area (sq ft)'),
    ('OverallQual',  'Overall Quality'),
    ('YearBuilt',    'Year Built'),
    ('TotRmsAbvGrd', 'Total Rooms Above Grade'),
]

for ax, (col, label) in zip(axes.flat, pairs):
    ax.scatter(train[col], train['SalePrice'], alpha=0.4, s=10, color='steelblue')
    ax.set_xlabel(label)
    ax.set_ylabel('log(SalePrice)')
    ax.set_title(f'{label} vs SalePrice')

plt.suptitle('Key Feature vs SalePrice', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Neighbourhood box plot
plt.figure(figsize=(18, 6))
order = train.groupby('Neighborhood')['SalePrice'].median().sort_values().index
sns.boxplot(data=train, x='Neighborhood', y='SalePrice', order=order, palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.title('SalePrice by Neighbourhood')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Feature Engineering

In [ ]:
# ── Combine train + test for consistent preprocessing ────────────────────────
y_train = train['SalePrice'].values
train_id = train['Id']
test_id  = test['Id']

all_data = pd.concat([train.drop(['Id', 'SalePrice'], axis=1),
                      test.drop('Id', axis=1)],
                     ignore_index=True)

print(f'all_data shape: {all_data.shape}')

### 3.1 Handle Missing Values

In [ ]:
# ── Categorical columns where NaN means 'None' (feature absent) ──────────────
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType',
]
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

# ── Numerical columns where NaN means 0 ──────────────────────────────────────
zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath',
    'MasVnrArea',
]
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

# ── LotFrontage – fill with median per neighbourhood ─────────────────────────
all_data['LotFrontage'] = (
    all_data.groupby('Neighborhood')['LotFrontage']
            .transform(lambda x: x.fillna(x.median()))
)

# ── Remaining categoricals – fill with mode ───────────────────────────────────
cat_cols = all_data.select_dtypes(include='object').columns
for col in cat_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# ── Remaining numericals – fill with median ───────────────────────────────────
num_cols = all_data.select_dtypes(include=[np.number]).columns
for col in num_cols:
    all_data[col] = all_data[col].fillna(all_data[col].median())

print(f'Remaining missing values: {all_data.isnull().sum().sum()}')

### 3.2 Feature Engineering

In [ ]:
# ── New features ──────────────────────────────────────────────────────────────
all_data['TotalSF']        = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBathrooms'] = (all_data['FullBath']
                              + 0.5 * all_data['HalfBath']
                              + all_data['BsmtFullBath']
                              + 0.5 * all_data['BsmtHalfBath'])
all_data['TotalPorchSF']   = (all_data['OpenPorchSF'] + all_data['3SsnPorch']
                              + all_data['EnclosedPorch'] + all_data['ScreenPorch']
                              + all_data['WoodDeckSF'])
all_data['HouseAge']       = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge']       = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['IsRemodeled']    = (all_data['YearBuilt'] != all_data['YearRemodAdd']).astype(int)
all_data['IsNew']          = (all_data['YearBuilt'] == all_data['YrSold']).astype(int)

# ── Convert ordinal quality columns to numeric ────────────────────────────────
quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
quality_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                'HeatingQC', 'KitchenQual', 'FireplaceQu',
                'GarageQual', 'GarageCond', 'PoolQC']
for col in quality_cols:
    all_data[col] = all_data[col].map(quality_map)

# ── Encode MSSubClass and MoSold as categorical ───────────────────────────────
all_data['MSSubClass'] = all_data['MSSubClass'].astype(str)
all_data['MoSold']     = all_data['MoSold'].astype(str)
all_data['YrSold']     = all_data['YrSold'].astype(str)

print('Feature engineering done.')
print(f'all_data shape: {all_data.shape}')

### 3.3 Skewness Correction for Numerical Features

In [ ]:
num_features = all_data.select_dtypes(include=[np.number]).columns
skewness = all_data[num_features].apply(lambda x: skew(x.dropna()))
skewed_features = skewness[abs(skewness) > 0.75].index

print(f'Number of skewed features (|skew| > 0.75): {len(skewed_features)}')

# Box-Cox / log1p transform
for col in skewed_features:
    all_data[col] = np.log1p(all_data[col].clip(lower=0))

print('Skewness correction applied.')

### 3.4 Encode Categorical Features

In [ ]:
all_data = pd.get_dummies(all_data)
print(f'all_data shape after encoding: {all_data.shape}')

### 3.5 Split Back into Train / Test

In [ ]:
n_train = len(y_train)
X_train = all_data.iloc[:n_train].values
X_test  = all_data.iloc[n_train:].values

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')

## 4. Model Training & Cross-Validation

In [ ]:
# ── Cross-validation helper ───────────────────────────────────────────────────
kf = KFold(n_splits=10, shuffle=True, random_state=42)

def rmsle_cv(model, X=X_train, y=y_train):
    scores = cross_val_score(
        model, X, y,
        scoring='neg_mean_squared_error',
        cv=kf
    )
    rmse = np.sqrt(-scores)
    return rmse

print('CV helper defined.')

### 4.1 Ridge Regression

In [ ]:
ridge = make_pipeline(RobustScaler(), Ridge(alpha=10.0, random_state=42))
score_ridge = rmsle_cv(ridge)
print(f'Ridge  RMSE: {score_ridge.mean():.4f} (± {score_ridge.std():.4f})')

### 4.2 Lasso Regression

In [ ]:
lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, max_iter=10000, random_state=42))
score_lasso = rmsle_cv(lasso)
print(f'Lasso  RMSE: {score_lasso.mean():.4f} (± {score_lasso.std():.4f})')

### 4.3 ElasticNet

In [ ]:
enet = make_pipeline(
    RobustScaler(),
    ElasticNet(alpha=0.0005, l1_ratio=0.9, max_iter=10000, random_state=42)
)
score_enet = rmsle_cv(enet)
print(f'ENet   RMSE: {score_enet.mean():.4f} (± {score_enet.std():.4f})')

### 4.4 Gradient Boosting

In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=3000, learning_rate=0.05,
    max_depth=4, max_features='sqrt',
    min_samples_leaf=15, min_samples_split=10,
    loss='huber', random_state=42
)
score_gbr = rmsle_cv(gbr)
print(f'GBR    RMSE: {score_gbr.mean():.4f} (± {score_gbr.std():.4f})')

### 4.5 XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    colsample_bytree=0.4603, gamma=0.0468,
    learning_rate=0.05, max_depth=3,
    min_child_weight=1.7817, n_estimators=2200,
    reg_alpha=0.4640, reg_lambda=0.8571,
    subsample=0.5213, verbosity=0,
    random_state=42, nthread=-1
)
score_xgb = rmsle_cv(xgb_model)
print(f'XGB    RMSE: {score_xgb.mean():.4f} (± {score_xgb.std():.4f})')

### 4.6 LightGBM

In [ ]:
lgbm_model = lgb.LGBMRegressor(
    objective='regression', num_leaves=5,
    learning_rate=0.05, n_estimators=720,
    max_bin=55, bagging_fraction=0.8,
    bagging_freq=5, feature_fraction=0.2319,
    feature_fraction_seed=9, bagging_seed=9,
    min_data_in_leaf=6, min_sum_hessian_in_leaf=11,
    random_state=42
)
score_lgbm = rmsle_cv(lgbm_model)
print(f'LGBM   RMSE: {score_lgbm.mean():.4f} (± {score_lgbm.std():.4f})')

## 5. Model Evaluation & Ensemble

In [ ]:
# ── CV score comparison ────────────────────────────────────────────────────────
results = {
    'Ridge'          : score_ridge,
    'Lasso'          : score_lasso,
    'ElasticNet'     : score_enet,
    'Gradient Boost' : score_gbr,
    'XGBoost'        : score_xgb,
    'LightGBM'       : score_lgbm,
}

means = {k: v.mean() for k, v in results.items()}
stds  = {k: v.std()  for k, v in results.items()}

print('\n{:<20} {:>12} {:>12}'.format('Model', 'Mean RMSE', 'Std RMSE'))
print('-' * 46)
for model in sorted(means, key=means.get):
    print('{:<20} {:>12.4f} {:>12.4f}'.format(model, means[model], stds[model]))

In [ ]:
# ── Bar chart comparison ───────────────────────────────────────────────────────
models_sorted = sorted(means, key=means.get)
mean_vals = [means[m] for m in models_sorted]
std_vals  = [stds[m]  for m in models_sorted]

plt.figure(figsize=(10, 5))
bars = plt.barh(models_sorted, mean_vals, xerr=std_vals,
                color='steelblue', alpha=0.8, capsize=4)
plt.xlabel('CV RMSE (log scale)')
plt.title('Model Comparison – 10-Fold CV RMSE')
plt.tight_layout()
plt.show()

### 5.1 Weighted Ensemble (Blending)

In [ ]:
# Fit all models on the full training set
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)
enet.fit(X_train, y_train)
gbr.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
lgbm_model.fit(X_train, y_train)

print('All models fitted.')

In [ ]:
def blend_predict(X):
    """Weighted average of all models."""
    preds = np.column_stack([
        ridge.predict(X),
        lasso.predict(X),
        enet.predict(X),
        gbr.predict(X),
        xgb_model.predict(X),
        lgbm_model.predict(X),
    ])
    # Equal weights – adjust as desired
    weights = np.array([0.10, 0.10, 0.10, 0.25, 0.25, 0.20])
    return preds @ weights

# Sanity check on training set
train_pred = blend_predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
print(f'Blend training RMSE: {train_rmse:.4f}')

## 6. Generate Submission

In [ ]:
# Predict on test set and invert log1p transform
test_pred_log = blend_predict(X_test)
test_pred     = np.expm1(test_pred_log)

submission = pd.DataFrame({
    'Id'       : test_id.values,
    'SalePrice': test_pred,
})

submission.to_csv('../data/submission.csv', index=False)
print('submission.csv saved.')
submission.head(10)

In [ ]:
# Distribution of predicted prices
plt.figure(figsize=(10, 5))
sns.histplot(test_pred, kde=True, color='coral')
plt.title('Distribution of Predicted Sale Prices')
plt.xlabel('Predicted SalePrice ($)')
plt.tight_layout()
plt.show()

print(f'Min  : ${test_pred.min():,.0f}')
print(f'Mean : ${test_pred.mean():,.0f}')
print(f'Max  : ${test_pred.max():,.0f}')

---

## Summary

| Step | What was done |
|------|---------------|
| EDA | Analysed target distribution, missing values, correlations, neighbourhood price trends |
| Preprocessing | Imputed missing values, created 7 engineered features, fixed skewness on 50+ columns, one-hot encoded categoricals |
| Models | Ridge, Lasso, ElasticNet, Gradient Boosting, XGBoost, LightGBM |
| Ensemble | Weighted blend of all 6 models |
| Output | `data/submission.csv` ready for Kaggle upload |

**Next steps**
- Hyperparameter tuning with `optuna` or `GridSearchCV`
- Stacking (meta-learner) instead of simple blending
- Deeper feature selection / removal of low-importance features